The purpose of this notebook is to deploy our model as part of the Product Search accelerator.  You may find this notebook on https://github.com/databricks-industry-solutions/product-search.

##Introduction

At this point we have a basic model as well as a tuned model, packaged with access to product embeddings and ready for deployment.  In this notebook, we will show how this model can be deployed using [Databricks Model Serving](https://docs.databricks.com/machine-learning/model-serving/index.html) so that applications can simply call a REST API to perform a search in real-time.

<img src='https://github.com/databricks-industry-solutions/product-search/raw/main/images/inference.png' width=800>

In [0]:
import mlflow
import os
import requests
import pandas as pd
import json
import time

In [0]:
%run "./00_Intro_and_Config"

##Step 1: Review Model Names

If we have successfully run the last two notebooks, we should have two models deployed to the MLflow model registry, each of which has been elevated to Production status.  You can select one or the other for Step 2:

In [0]:
print(f"Basic Model: {config['basic_model_name']}")
print(f"Tuned Model: {config['tuned_model_name']}")

In [0]:
model_name = config['tuned_model_name']

# identify model version in registry
model_version = mlflow.tracking.MlflowClient().get_latest_versions(name = model_name, stages = ["Production"])[0].version

##Step 2: Deploy Model to Model Serving Endpoint

To deploy our model, we need to reconfigure our Databricks workspace for Machine Learning.  We can do this by clicking on the drop-down at the top of the left-hand navigation bar and selecting *Machine Learning*.
</p>

<img src='https://brysmiwasb.blob.core.windows.net/demos/images/search_change_workspace.png'>

Once we've done that, we should be able to select *Serving* from that same left-hand navigation bar.

Within the Serving Endpoints page, click on the *Create Serving Endpoint* button.  Give your endpoint a name, select your model - it may help to start typing the model name to limit the search - and then select the model version.  Select the compute size based on the number of requests expected and select/deselect the *scale to zero* option based on whether you want the service to scale down completely during a sustained period of inactivity.  (Spinning back up from zero does take a little time once a request has been received.)
</p>

<img src='https://brysmiwasb.blob.core.windows.net/demos/images/search_create_serving_endpoint2.png' width=90%>

Click the *Create serving endpoint* button to deploy the endpoint and monitor the deployment process until the *Serving Endpoint State* is *Ready*:
</p>

<img src='https://brysmiwasb.blob.core.windows.net/demos/images/search_endpoint_ready.png' width=90%>


Before leaving this page, be sure to click the *Query Endpoint* button in the upper right-hand corner.  In the resulting pane, click on the *Python* tab and copy the displayed code to the cell below:
</p>

<img src='https://brysmiwasb.blob.core.windows.net/demos/images/search_query_endpoint.PNG' width=50%>

Alternatively, we can use the API instead of the UI to [set up the model serving endpoint](https://docs.databricks.com/machine-learning/model-serving/create-manage-serving-endpoints.html#create-model-serving-endpoints). The effect of the following 3 blocks of code is equivalent to the steps described above. We provide this option to showcase automation and to make sure that this notebook can be consistently executed end-to-end without requiring manual intervention.

To use the Databricks API, you need to create environmental variables named *DATABRICKS_URL* and *DATABRICKS_TOKEN* which must be your workspace url and a valid [personal access token](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/api/latest/authentication). We have retrieved and set these values up for you in notebook *00* as part of the *config* setting.

In [0]:
%run ./util/create-update-serving-endpoint

In [0]:
served_models = [
    {
      "name": "Product-Search",
      "model_name": model_name,
      "model_version": model_version,
      "workload_size": "Medium",
      "scale_to_zero_enabled": True
    }
]
traffic_config = {"routes": [{"served_model_name": "Product-Search", "traffic_percentage": "100"}]}

In [0]:
# kick off endpoint creation/update
if not endpoint_exists(config['serving_endpoint_name']):
  create_endpoint(config['serving_endpoint_name'], served_models)
else:
  update_endpoint(config['serving_endpoint_name'], served_models)

##Step 3: Test the Model Serving Endpoint

With the code for testing our endpoint in the cell below, we can now prepare to submit data against our endpoint:

In [0]:
import os
import requests
import numpy as np
import pandas as pd
import json

endpoint_url = f"""{config['databricks url']}/serving-endpoints/{config['serving_endpoint_name']}/invocations"""

def create_tf_serving_json(data):
  return {'inputs': {name: data[name].tolist() for name in data.keys()} if isinstance(data, dict) else data.tolist()}

def score_model(dataset):
  url = endpoint_url
  headers = {'Authorization': f'Bearer {os.environ.get("DATABRICKS_TOKEN")}', 'Content-Type': 'application/json'}
  ds_dict = {'dataframe_split': dataset.to_dict(orient='split')} if isinstance(dataset, pd.DataFrame) else create_tf_serving_json(dataset)
  data_json = json.dumps(ds_dict, allow_nan=True)
  response = requests.request(method='POST', headers=headers, url=url, data=data_json)
  if response.status_code != 200:
    raise Exception(f'Request failed with status {response.status_code}, {response.text}')

  return response.json()

And now we can test the endpoint:

In [0]:
score_model( 
  pd.DataFrame({'query':['kid-proof rug']})
)

© 2023 Databricks, Inc. All rights reserved. The source in this notebook is provided subject to the Databricks License. All included or referenced third party libraries are subject to the licenses set forth below.

| library                                | description             | license    | source                                              |
|----------------------------------------|-------------------------|------------|-----------------------------------------------------|
|  WANDS | Wayfair product search relevance data | MIT  | https://github.com/wayfair/WANDS   |
| langchain | Building applications with LLMs through composability | MIT  |   https://pypi.org/project/langchain/ |
| chromadb | An open source embedding database |  Apache |  https://pypi.org/project/chromadb/  |
| sentence-transformers | Compute dense vector representations for sentences, paragraphs, and images | Apache 2.0 |https://pypi.org/project/sentence-transformers/ |